In [8]:
import optuna
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pandas as pd
import datetime
import io
import contextlib
import json



In [9]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available: 1


In [10]:
NUM_CLASSES = 100
INPUT_SHAPE = (32, 32, 3)
MAX_EPOCHS = 50
VALIDATION_SPLIT = 0.2

In [11]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data(label_mode='fine')

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

In [12]:
# Global list to store results
trial_log = []


def objective(trial):
    K.clear_session()

    # 🎯 Hyperparameter suggestions
    lr = trial.suggest_categorical("lr", [1e-5, 1e-4, 1e-3])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])
    dropout_rate = trial.suggest_categorical("dropout", [0.2, 0.3, 0.4])

    # Filters
    filters_1 = trial.suggest_categorical("filters_1", [64, 128, 256])  # always included
    filters_2 = trial.suggest_categorical("filters_2", [0, 64, 128, 256])
    filters_3 = trial.suggest_categorical("filters_3", [0, 64, 128, 256])

    dense_units = trial.suggest_categorical("dense_units", [128, 256, 512])
    batch_size = trial.suggest_categorical("batch_size", [64, 128])

    optimizer = Adam(learning_rate=lr) if optimizer_name == "Adam" else RMSprop(learning_rate=lr)
    
    print(f"\nTrial {trial.number} Hyperparameters:")
    print(f"  Learning rate: {lr}")
    print(f"  Optimizer:     {optimizer_name}")
    print(f"  Dropout:       {dropout_rate}")
    print(f"  Filters:       [{filters_1}, {filters_2}, {filters_3}]")
    print(f"  Dense units:   {dense_units}")
    print(f"  Batch size:    {batch_size}")

    # 🧱 Build model
    model = Sequential()

    # Conv Block 1 (always included)
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu', input_shape=INPUT_SHAPE))
    model.add(BatchNormalization())
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(dropout_rate))

    # Conv Block 2 (optional)
    if filters_2 > 0:
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Conv Block 3 (optional)
    if filters_3 > 0:
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Dense block
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

    # Train the model
    history = model.fit(
        x_train, y_train_cat,
        validation_split=VALIDATION_SPLIT,
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        verbose=1,
        callbacks=[early_stopping, reduce_lr]
    )

    val_acc = history.history['val_accuracy'][-1]
    test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
    
    with io.StringIO() as buf, contextlib.redirect_stdout(buf):
        model.summary()
        model_summary_str = buf.getvalue()


    trial_log.append({
        'trial': trial.number,
        'lr': lr,
        'optimizer': optimizer_name,
        'dropout': dropout_rate,
        'filters_1': filters_1,
        'filters_2': filters_2,
        'filters_3': filters_3,
        'dense_units': dense_units,
        'batch_size': batch_size,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'epochs_ran': len(history.history['accuracy']),
        'timestamp': datetime.datetime.now().isoformat(),
        'model_summary': model_summary_str 
    })

    return val_acc


In [13]:
try:
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=25, show_progress_bar=True)

except KeyboardInterrupt:
    print("\nOptimization interrupted. Saving progress...")

finally:
    # Save to CSV and JSON no matter what
    pd.DataFrame(trial_log).to_csv("optuna_trials.csv", index=False)

    with open("optuna_trials_with_summaries.json", "w") as f:
        json.dump(trial_log, f, indent=2)

    print("Trials saved to 'optuna_trials.csv' and 'optuna_trials_with_summaries.json'")


[I 2025-03-27 01:01:20,824] A new study created in memory with name: no-name-9f6d9f40-11b4-4a10-89e8-6b3717925b8d
  0%|          | 0/25 [00:00<?, ?it/s]


Trial 0 Hyperparameters:
  Learning rate: 0.001
  Optimizer:     Adam
  Dropout:       0.2
  Filters:       [64, 256, 0]
  Dense units:   256
  Batch size:    128


/Users/awabakram/git/cognitive_robotics_coursework/myenv/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 39s 120ms/step - accuracy: 0.0135 - loss: 61.1166 - val_accuracy: 0.0224 - val_loss: 28.9553 - learning_rate: 0.0010
Epoch 2/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 118ms/step - accuracy: 0.0279 - loss: 82.6672 - val_accuracy: 0.0256 - val_loss: 82.0060 - learning_rate: 0.0010
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 123ms/step - accuracy: 0.0358 - loss: 121.8271 - val_accuracy: 0.0624 - val_loss: 51.5703 - learning_rate: 0.0010
Epoch 4/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.0444 - loss: 102.8914
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
313/313 ━━━━━━━━━━━━━━━━━━━━ 45s 144ms/step - accuracy: 0.0444 - loss: 102.8768 - val_accuracy: 0.0677 - val_loss: 59.4357 - learning_rate: 0.0010
Epoch 5/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 57s 182ms/step - accuracy: 0.0591 - loss: 70.2979 - val_accuracy: 0.0938 - val_loss: 24.0220 - learning_rate: 5.0000e-04
Epoch 6/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 52s 167ms/step

Model: "sequential"

...

 Optimizer params: 2,585,034 (9.86 MB)

Best trial: 3. Best value: 0.5356:  28%|██▊       | 7/25 [7:40:34<14:48:36, 2962.03s/it]

[I 2025-03-27 08:41:55,414] Trial 6 finished with value: 0.5321999788284302 and parameters: {'lr': 0.001, 'optimizer': 'Adam', 'dropout': 0.3, 'filters_1': 128, 'filters_2': 128, 'filters_3': 128, 'dense_units': 256, 'batch_size': 128}. Best is trial 3 with value: 0.5356000065803528.

Trial 7 Hyperparameters:
  Learning rate: 0.001
  Optimizer:     RMSprop
  Dropout:       0.4
  Filters:       [256, 64, 128]
  Dense units:   512
  Batch size:    64
Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 122s 191ms/step - accuracy: 0.0135 - loss: 14.0192 - val_accuracy: 0.0388 - val_loss: 6.7504 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 120s 192ms/step - accuracy: 0.0372 - loss: 14.5599 - val_accuracy: 0.0434 - val_loss: 9.1901 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 121s 193ms/step - accuracy: 0.0593 - loss: 10.0677 - val_accuracy: 0.0754 - val_loss: 5.8297 - learning_rate: 0.0010
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 121s 194ms/step - accuracy: 0.0853 -

Best trial: 3. Best value: 0.5356:  28%|██▊       | 7/25 [9:09:58<23:34:12, 4714.00s/it]

[W 2025-03-27 10:11:18,854] Trial 7 failed with parameters: {'lr': 0.001, 'optimizer': 'RMSprop', 'dropout': 0.4, 'filters_1': 256, 'filters_2': 64, 'filters_3': 128, 'dense_units': 512, 'batch_size': 64} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/awabakram/git/cognitive_robotics_coursework/myenv/lib/python3.9/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/jm/k37qm_7n7dj5fp2rnwhd32y80000gn/T/ipykernel_17066/2632051870.py", line 72, in objective
    history = model.fit(
  File "/Users/awabakram/git/cognitive_robotics_coursework/myenv/lib/python3.9/site-packages/keras/src/utils/traceback_utils.py", line 117, in error_handler
    return fn(*args, **kwargs)
  File "/Users/awabakram/git/cognitive_robotics_coursework/myenv/lib/python3.9/site-packages/keras/src/backend/tensorflow/trainer.py", line 371, in fit
    logs = self.train_function(iterator)
  File "/Us